## Problem Statement

### Context

AllLife Bank is a US bank that has a growing customer base. The majority of these customers are liability customers (depositors) with varying sizes of deposits. The number of customers who are also borrowers (asset customers) is quite small, and the bank is interested in expanding this base rapidly to bring in more loan business and in the process, earn more through the interest on loans. In particular, the management wants to explore ways of converting its liability customers to personal loan customers (while retaining them as depositors).

A campaign that the bank ran last year for liability customers showed a healthy conversion rate of over 9% success. This has encouraged the retail marketing department to devise campaigns with better target marketing to increase the success ratio.

You as a Data scientist at AllLife bank have to build a model that will help the marketing department to identify the potential customers who have a higher probability of purchasing the loan.

### Objective

To predict whether a liability customer will buy personal loans, to understand which customer attributes are most significant in driving purchases, and identify which segment of customers to target more.

### Data Dictionary
* `ID`: Customer ID
* `Age`: Customer’s age in completed years
* `Experience`: #years of professional experience
* `Income`: Annual income of the customer (in thousand dollars)
* `ZIP Code`: Home Address ZIP code.
* `Family`: the Family size of the customer
* `CCAvg`: Average spending on credit cards per month (in thousand dollars)
* `Education`: Education Level. 1: Undergrad; 2: Graduate;3: Advanced/Professional
* `Mortgage`: Value of house mortgage if any. (in thousand dollars)
* `Personal_Loan`: Did this customer accept the personal loan offered in the last campaign? (0: No, 1: Yes)
* `Securities_Account`: Does the customer have securities account with the bank? (0: No, 1: Yes)
* `CD_Account`: Does the customer have a certificate of deposit (CD) account with the bank? (0: No, 1: Yes)
* `Online`: Do customers use internet banking facilities? (0: No, 1: Yes)
* `CreditCard`: Does the customer use a credit card issued by any other Bank (excluding All life Bank)? (0: No, 1: Yes)

## Importing necessary libraries

In [5]:
# Installing the libraries with the specified version.
#!pip install numpy==1.25.2 pandas==1.5.3 matplotlib==3.7.1 seaborn==0.13.1 scikit-learn==1.2.2 sklearn-pandas==2.2.0 -q --user
#!pip install numpy==2.0.1 pandas==2.2.1 matplotlib==3.8.2 seaborn==0.13.2 scikit-learn==1.4.1.post1 sklearn-pandas==2.2.0
!pip install numpy==1.26.4 pandas==2.2.1 matplotlib==3.8.2 seaborn==0.13.2 scikit-learn==1.4.1.post1 sklearn-pandas==2.2.0 -q --user

**Note**:

1. After running the above cell, kindly restart the notebook kernel (for Jupyter Notebook) or runtime (for Google Colab), write the relevant code for the project from the next cell, and run all cells sequentially from the next cell.

2. On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [7]:
# Libraries to help with reading and manipulating data
import pandas as pd
import numpy as np

# libaries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Library to split data
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# To build model for prediction
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn import tree

#for preprocessing
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# To get diferent metric scores
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    recall_score,
    precision_score,
    confusion_matrix,
    roc_auc_score,
)

# to suppress unnecessary warnings
import warnings
warnings.filterwarnings("ignore")
from sklearn.compose import ColumnTransformer

## Loading the dataset

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Original_data = pd.read_csv('/content/drive/MyDrive/AIML-UTA-PY/Loan_Modelling.csv')
data = Original_data.copy()
display(data.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,ID,Age,Experience,Income,ZIPCode,Family,CCAvg,Education,Mortgage,Personal_Loan,Securities_Account,CD_Account,Online,CreditCard
0,1,25,1,49,91107,4,1.6,1,0,0,1,0,0,0
1,2,45,19,34,90089,3,1.5,1,0,0,1,0,0,0
2,3,39,15,11,94720,1,1.0,1,0,0,0,0,0,0
3,4,35,9,100,94112,1,2.7,2,0,0,0,0,0,0
4,5,35,8,45,91330,4,1.0,2,0,0,0,0,0,1


## Data Overview

* Observations
* Sanity checks

In [11]:
data.describe(include='all').T

,count,mean,std,min,25%,50%,75%,max
ID,5000.0,2500.500000,1443.520003,1.0,1250.75,2500.5,3750.25,5000.0
Age,5000.0,45.338400,11.463166,23.0,35.00,45.0,55.00,67.0
Experience,5000.0,20.104600,11.467954,-3.0,10.00,20.0,30.00,43.0
Income,5000.0,73.774200,46.033729,8.0,39.00,64.0,98.00,224.0
ZIPCode,5000.0,93169.257000,1759.455086,90005.0,91911.00,93437.0,94608.00,96651.0
Family,5000.0,2.396400,1.147663,1.0,1.00,2.0,3.00,4.0
CCAvg,5000.0,1.937938,1.747659,0.0,0.70,1.5,2.50,10.0
Education,5000.0,1.881000,0.839869,1.0,1.00,2.0,3.00,3.0
Mortgage,5000.0,56.498800,101.713802,0.0,0.00,0.0,101.00,635.0
Personal_Loan,5000.0,0.096000,0.294621,0.0,0.00,0.0,0.00,1.0


In [12]:
print("first 5 Rows:")
data.head()

first 5 Rows:


,ID,Age,Experience,Income,ZIPCode,Family,CCAvg,Education,Mortgage,Personal_Loan,Securities_Account,CD_Account,Online,CreditCard
0,1,25,1,49,91107,4,1.6,1,0,0,1,0,0,0
1,2,45,19,34,90089,3,1.5,1,0,0,1,0,0,0
2,3,39,15,11,94720,1,1.0,1,0,0,0,0,0,0
3,4,35,9,100,94112,1,2.7,2,0,0,0,0,0,0
4,5,35,8,45,91330,4,1.0,2,0,0,0,0,0,1


In [13]:
print("last 5 Rows:")
data.tail()

last 5 Rows:


,ID,Age,Experience,Income,ZIPCode,Family,CCAvg,Education,Mortgage,Personal_Loan,Securities_Account,CD_Account,Online,CreditCard
4995,4996,29,3,40,92697,1,1.9,3,0,0,0,0,1,0
4996,4997,30,4,15,92037,4,0.4,1,85,0,0,0,1,0
4997,4998,63,39,24,93023,2,0.3,3,0,0,0,0,0,0
4998,4999,65,40,49,90034,3,0.5,2,0,0,0,0,1,0
4999,5000,28,4,83,92612,3,0.8,1,0,0,0,0,1,1


In [14]:
# Number of rows and columns in the data using the shape attribute of the DataFrame.
rows, columns = data.shape


print(f'Number of Rows: {rows}')
print(f'Number of Columns: {columns}')

Number of Rows: 5000
Number of Columns: 14


In [15]:
# Determine if there is any missing data.
missing_values = data.isnull().sum()

# Check for nulls and NaN values in the dataset.
print('Null values ' + str(data.isnull().sum()) + '\n')
print('NaN values ' + str(data.isna().sum()) + '\n')

# Output if there are any missing data points in the dataset.
if missing_values.sum() > 0:
    print('There are missing data points in the Personal Loan dataset.')
else:
    print('There are no missing data points in the Personal Load dataset.')

Null values ID                    0
Age                   0
Experience            0
Income                0
ZIPCode               0
Family                0
CCAvg                 0
Education             0
Mortgage              0
Personal_Loan         0
Securities_Account    0
CD_Account            0
Online                0
CreditCard            0
dtype: int64

NaN values ID                    0
Age                   0
Experience            0
Income                0
ZIPCode               0
Family                0
CCAvg                 0
Education             0
Mortgage              0
Personal_Loan         0
Securities_Account    0
CD_Account            0
Online                0
CreditCard            0
dtype: int64

There are no missing data points in the Personal Load dataset.


In [16]:
# Drop the ID column from the dataset.
if 'ID' in data.columns:
    data.drop(['ID'], axis=1, inplace=True)

# Retrieve the first 5 rows of the dataset after dropping the ID column.
print('First 5 rows of the dataset after dropping the ID column:')
data.head()

First 5 rows of the dataset after dropping the ID column:


,Age,Experience,Income,ZIPCode,Family,CCAvg,Education,Mortgage,Personal_Loan,Securities_Account,CD_Account,Online,CreditCard
0,25,1,49,91107,4,1.6,1,0,0,1,0,0,0
1,45,19,34,90089,3,1.5,1,0,0,1,0,0,0
2,39,15,11,94720,1,1.0,1,0,0,0,0,0,0
3,35,9,100,94112,1,2.7,2,0,0,0,0,0,0
4,35,8,45,91330,4,1.0,2,0,0,0,0,0,1


In [17]:
# Define bins and labels
bins = [0, 20, 50, 100, 500, float('inf')]
labels = ['0–20K', '20K–50K', '50K-100k','100k-500k', '500k+']

# Create a new column with the mortgage range
data['Mortgage_Range'] = pd.cut(data['Mortgage'], bins=bins, labels=labels, right=False)

# Count how many fall into each range
mortgage_range_distribution = data['Mortgage_Range'].value_counts().sort_index()

print('Distribution of Mortgages by Range:')
print(mortgage_range_distribution)

Distribution of Mortgages by Range:
Mortgage_Range
0–20K        3462
20K–50K         0
50K-100k      270
100k-500k    1242
500k+          26
Name: count, dtype: int64


In [18]:
# Map Education codes to labels
education_map = {
    1: 'Undergrad',
    2: 'Graduate',
    3: 'Advanced/Professional'
}

data['Education_Label'] = data['Education'].map(education_map)

# Create the cross-tab
zip_education_crosstab = pd.crosstab(data['ZIPCode'], data['Education_Label'])
# Keep only ZIP codes with total count > 20
zipcode_education_filtered = zip_education_crosstab[zip_education_crosstab.sum(axis=1) > 20]

# Sort by ZIP code (optional)
zipcode_education_filtered = zipcode_education_filtered.sort_index()

# Display
print('Cross-tabulation of ZIP code and Education (Total count > 20):')
print(zipcode_education_filtered.to_string())

Cross-tabulation of ZIP code and Education (Total count > 20):
Education_Label  Advanced/Professional  Graduate  Undergrad
ZIPCode                                                    
90024                               18        12         20
90089                               10        14         22
90095                               26        20         25
90245                               21        12         17
91107                               12         5          8
91320                               17        14         22
91330                               12        15         19
91380                               14         4          4
91711                               19        18         15
91768                                7         9          5
92028                               10        11         11
92037                               17        12         25
92093                               15        17         19
92121                                

In [19]:

# Checking to see if there is a correlation between Education level and Mortgage amount.
# Calculate the correlation between 'Mortgage' and 'Education'
correlation = data['Mortgage'].corr(data['Education'])

print(f"The correlation between Mortgage and Education is: {correlation}")

The correlation between Mortgage and Education is: -0.033327124629950425


In [20]:

# Checking to see if there is a correlation between Education level and Mortgage amount.
# Calculate the correlation between 'Mortgage' and 'Education'
correlation = data['Mortgage'].corr(data['Income'])

print(f"The correlation between Mortgage and Income is: {correlation}")

The correlation between Mortgage and Income is: 0.2068062278031723


### **Data Quality Observation**

The dataset contains 5,000 entries with 14 columns capturing demographic, financial, and behavioral attributes of customers.

No missing values are present in any column, indicating complete data coverage across all records.

Data types are appropriate and consistent:

Mostly integer types for categorical and count data (e.g., Age, Income, Education).

One float column for average credit card spending (CCAvg).

The absence of nulls reduces potential bias and errors associated with handling missing data, ensuring reliable insights and model performance.

### **Observation on ZIP Code and Education Analysis**

The cross-tabulation of ZIP codes against education levels reveals varying distributions of educational attainment across different geographic areas.

Some ZIP codes (e.g., 94720, 95616, and 94305) show higher counts across all education levels, indicating these areas have a larger population or more customers represented in the dataset.

Other ZIP codes have smaller counts, suggesting fewer customers or less representation.

This spatial variation in education could inform targeted marketing or financial product strategies, tailoring offers based on regional education profiles.

### **Observation on Mortgage Distribution**

The majority of customers have mortgages in the 0–20K range, likely indicating many customers have no or low mortgage balances.

There is a noticeable absence of mortgages in the 20K–50K range, which might be due to binning strategy or data characteristics that could warrant further review.

Substantial numbers of customers hold mortgages in the 50K–100K and 100K–500K ranges, reflecting a diverse mortgage portfolio.

A small portion of customers have mortgages exceeding 500K, representing high-value mortgage holders.

Understanding these mortgage ranges helps identify segments with differing credit exposure and could guide risk assessment and product design.

## Exploratory Data Analysis.

- EDA is an important part of any project involving data.
- It is important to investigate and understand the data better before building a model with it.
- A few questions have been mentioned below which will help you approach the analysis in the right manner and generate insights from the data.
- A thorough analysis of the data, in addition to the questions mentioned below, should be done.

**Questions**:

1. What is the distribution of mortgage attribute? Are there any noticeable patterns or outliers in the distribution?
2. How many customers have credit cards?
3. What are the attributes that have a strong correlation with the target attribute (personal loan)?
4. How does a customer's interest in purchasing a loan vary with their age?
5. How does a customer's interest in purchasing a loan vary with their education?

## Data Preprocessing

* Missing value treatment
* Feature engineering (if needed)
* Outlier detection and treatment (if needed)
* Preparing data for modeling
* Any other preprocessing steps (if needed)

## Model Building

### Model Evaluation Criterion

*


### Model Building

## Model Performance Improvement

## Model Performance Comparison and Final Model Selection

## Actionable Insights and Business Recommendations


* What recommedations would you suggest to the bank?

___